NEON CONSOLE LINK - https://console.neon.tech/app/projects/

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

In [0]:
jdbc_url = f"jdbc:{os.getenv('DB_TYPE')}://{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"

connection_props = {
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "driver": os.getenv("DB_DRIVER")
}

In [0]:
def get_db_type(jdbc_url):
    url = jdbc_url.lower()

    if "postgresql" in url:
        return "postgres"
    elif "mysql" in url or "mariadb" in url:
        return "mysql"
    elif "sqlserver" in url:
        return "sqlserver"
    elif "oracle" in url:
        return "oracle"
    elif "snowflake" in url:
        return "snowflake"
    elif "databricks" in url or "hive" in url:
        return "spark"
    elif "redshift" in url:
        return "redshift"
    elif "db2" in url:
        return "db2"
    else:
        return "unknown"


def get_tables_query(db_type):

    # ✅ POSTGRES / REDSHIFT
    if db_type in ["postgres", "redshift"]:
        return """
        (SELECT table_schema, table_name, table_type
         FROM information_schema.tables
         WHERE table_schema NOT IN (
             'pg_catalog',
             'information_schema',
             'pg_toast'
         )
        ) as tables
        """

    # ✅ MYSQL / MARIADB
    elif db_type == "mysql":
        return """
        (SELECT table_schema, table_name, table_type
         FROM information_schema.tables
         WHERE table_schema NOT IN (
             'information_schema',
             'mysql',
             'performance_schema',
             'sys'
         )
        ) as tables
        """

    # ✅ SQL SERVER
    elif db_type == "sqlserver":
        return """
        (SELECT TABLE_SCHEMA as table_schema,
                TABLE_NAME as table_name,
                TABLE_TYPE as table_type
         FROM INFORMATION_SCHEMA.TABLES
         WHERE TABLE_SCHEMA NOT IN (
             'INFORMATION_SCHEMA',
             'sys'
         )
        ) as tables
        """

    # ✅ ORACLE
    elif db_type == "oracle":
        return """
        (SELECT owner as table_schema,
                table_name,
                'BASE TABLE' as table_type
         FROM all_tables
         WHERE owner NOT IN (
             'SYS',
             'SYSTEM',
             'XDB',
             'CTXSYS',
             'MDSYS',
             'ORDSYS',
             'OUTLN'
         )
        ) as tables
        """

    # ✅ SNOWFLAKE
    elif db_type == "snowflake":
        return """
        (SELECT table_schema, table_name, table_type
         FROM information_schema.tables
         WHERE table_schema NOT IN (
             'INFORMATION_SCHEMA'
         )
        ) as tables
        """

    # ✅ DATABRICKS / HIVE
    elif db_type == "spark":
        return """
        (SELECT database as table_schema,
                tableName as table_name,
                'BASE TABLE' as table_type
         FROM system.information_schema.tables
         WHERE database NOT IN (
             'information_schema'
         )
        ) as tables
        """

    # ✅ IBM DB2
    elif db_type == "db2":
        return """
        (SELECT tabschema as table_schema,
                tabname as table_name,
                type as table_type
         FROM syscat.tables
         WHERE tabschema NOT IN (
             'SYSIBM',
             'SYSCAT',
             'SYSSTAT',
             'SYSFUN',
             'SYSIBMADM'
         )
        ) as tables
        """

    else:
        raise Exception("Unsupported DB")




db_type = get_db_type(jdbc_url)
tables_df = spark.read.jdbc(
    url=jdbc_url,
    table=get_tables_query(db_type),
    properties=connection_props
)
tables_df = tables_df.filter("table_schema IS NOT NULL")


In [0]:
display(tables_df)

In [0]:
display(tables_df)

In [0]:
schemas = [row['table_schema'] for row in tables_df.select("table_schema").distinct().collect()]

tables = [
    f"{row['table_schema']}.{row['table_name']}"
    for row in tables_df.select("table_schema", "table_name").collect()
]

catalog_name = "test_catalog3"

tables_with_catalog = [
    f"{catalog_name}.{t}"
    for t in tables
]


In [0]:
row_counts = []

for table in tables:
    query = f"(SELECT COUNT(*) as row_count FROM {table}) as t"
    
    df = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=connection_props
    )
    
    count = df.collect()[0]["row_count"]
    
    row_counts.append((table, count))

# Convert to DataFrame
row_count_df = spark.createDataFrame(row_counts, ["table", "row_count"])


In [0]:
large_table_batch_size=10
table_size_limit=200

In [0]:
large_tables = [table for table, count in row_counts if count > table_size_limit]
small_tables = [table for table, count in row_counts if count <=table_size_limit]

In [0]:
large_tables_full = [
    tgt for src, tgt in zip(tables, tables_with_catalog)
    if src in large_tables
]

small_tables_full = [
    tgt for src, tgt in zip(tables, tables_with_catalog)
    if src in small_tables
]

In [0]:
print("Database type: ",db_type)
print("Schemas: ", schemas)
print("Tables:", tables)
print("Full large tables: ",large_tables_full)
print("Full small tables: ",small_tables_full)

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

In [0]:
for i in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{i}")

In [0]:
for i in large_tables_full:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {i}")


for i in small_tables_full:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {i}")

load small tables


In [0]:
for tgt_table, src_table in zip(small_tables_full, small_tables):
    df=spark.read.jdbc(
        url=jdbc_url,
        table=src_table,
        properties=connection_props
    )
    df.write.\
        format("delta")\
        .mode("overwrite")\
        .option("mergeSchema","true")\
        .saveAsTable(tgt_table)
    print(f"Data migrated from {db_type}-{src_table} to {tgt_table}")

load large tables

In [0]:
props = connection_props.copy()  # copy your existing JDBC properties
props["fetchsize"] = f"{large_table_batch_size}"

for tgt_table, src_table in zip(large_tables_full, large_tables):
    df=spark.read.jdbc(
        url=jdbc_url,
        table=src_table,
        properties=props
    )
    df.write.\
        format("delta")\
        .mode("overwrite")\
        .option("mergeSchema","true")\
        .saveAsTable(tgt_table)
    print(f"Data migrated from {db_type}-{src_table} to {tgt_table}")

In [0]:
%sql
--DROP CATALOG test_catalog3 CASCADE;